<a href="https://colab.research.google.com/github/afianas/DermaScan/blob/main/Dermascan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from google.colab import drive, files
import os
import shutil
from PIL import Image
import matplotlib.pyplot as plt

# --- ACTION REQUIRED BY TEAM MEMBER ---
# Create a folder in your Drive called 'Dermascan_Project'
# and put your kaggle.json there.
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Dermascan_Project'
# ---------------------------------------

# Define sub-paths automatically
checkpoint_path = os.path.join(BASE_PATH, 'acne_model_best.keras') # Use .keras format
processed_base = 'Organized_Acne_Data'

# Initialize the Checkpoint object
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,
    monitor='val_accuracy',
    verbose=1
)

In [ ]:
# Setup Kaggle (Uses the file from your Drive instead of manual upload)
kaggle_config = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_config, exist_ok=True)
shutil.copy(os.path.join(BASE_PATH, 'kaggle.json'), os.path.join(kaggle_config, 'kaggle.json'))
os.chmod(os.path.join(kaggle_config, 'kaggle.json'), 0o600)

!kaggle datasets download -d lexuanhieu131297/acne-severity-classification
!unzip -q acne-severity-classification.zip -d acne_raw

In [ ]:
import os
img_folder = 'acne_raw/JPEGImages'
if os.path.exists(img_folder):
    print(f"Success! Total images found: {len(os.listdir(img_folder))}")
else:
    print("Error: Could not find the JPEGImages folder. Double-check the unzip path.")

In [ ]:
import os

# 1. Check if the zip file exists
if os.path.exists('acne-severity-classification.zip'):
    print("✅ Zip file found.")
else:
    print("❌ Zip file NOT found. Re-run the '!kaggle datasets download' cell.")

# 2. List everything in the 'acne_raw' folder
print("\n--- Folder Structure inside 'acne_raw' ---")
if os.path.exists('acne_raw'):
    for root, dirs, files in os.walk('acne_raw'):
        # Just show the first two levels to keep it clean
        level = root.replace('acne_raw', '').count(os.sep)
        if level < 2:
            print(f"{'  ' * level}{os.path.basename(root)}/")
else:
    print("❌ 'acne_raw' folder does not exist.")

In [ ]:
import os
from PIL import Image

# 1. Use the BASE_PATH we defined at the start of the program
# This ensures it works for you and your friend's Drive
processed_base = 'Organized_Acne_Data'
classes = ['Mild', 'Moderate', 'Severe']

for cls in classes:
    os.makedirs(os.path.join(processed_base, cls), exist_ok=True)

# 2. Update these paths to be relative to your unzipped Kaggle folder
raw_img_path = 'acne_raw/Classification/JPEGImages'
label_file = 'acne_raw/Classification/NNEW_trainval_0.txt'

def sort_and_resize(source_path, label_path):
    if not os.path.exists(label_path):
        print(f"❌ Error: Label file not found at {label_path}")
        return

    with open(label_path, 'r') as f:
        lines = f.readlines()
        print(f"Processing {len(lines)} images...")

        for line in lines:
            parts = line.split()
            img_name = parts[0]
            label = int(parts[1])

            # Map to 3 clinical levels as per base paper
            if label == 0:
                target_folder = 'Mild'
            elif label in [1, 2]:
                target_folder = 'Moderate'
            else:
                target_folder = 'Severe'

            src = os.path.join(source_path, img_name)
            dest = os.path.join(processed_base, target_folder, img_name)

            if os.path.exists(src):
                with Image.open(src) as img:
                    # Standardizing to 224x224 for MobileNetV2
                    img.convert('RGB').resize((224, 224)).save(dest)
            else:
                # Useful for debugging shared drives
                print(f"⚠️ Warning: {img_name} not found in source.")

# Execute the function
sort_and_resize(raw_img_path, label_file)
print("✅ Sorting and resizing complete!")

Run this code to display one random image from your "Severe" folder to confirm the 224×224 resizing worked

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

folder = 'Organized_Acne_Data/Severe'
first_img = os.listdir(folder)[0]
img = mpimg.imread(os.path.join(folder, first_img))
plt.imshow(img)
plt.title(f"Processed Image Size: {img.shape}")
plt.show()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
import matplotlib.pyplot as plt
import os

# 1. Setup the Augmentation Rules (based on the paper)
datagen = ImageDataGenerator(
    rotation_range=40,      # Rotates for different facial orientations [cite: 168]
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,         # Simulates different distances from subject [cite: 169]
    horizontal_flip=True,   # Mirrors for varied perspectives [cite: 168]
    brightness_range=[0.5, 1.5], # Accommodates different lighting [cite: 169]
    fill_mode='nearest'
)

# 2. Pick one image to test
folder = 'Organized_Acne_Data/Moderate'
img_path = os.path.join(folder, os.listdir(folder)[0])
img = load_img(img_path)
x = img_to_array(img).reshape((1,) + img_to_array(img).shape)

# 3. Generate and plot 9 variations
plt.figure(figsize=(12, 12))
i = 0
for batch in datagen.flow(x, batch_size=1):
    plt.subplot(3, 3, i + 1)
    plt.imshow(batch[0].astype('uint8'))
    plt.axis('off')
    i += 1
    if i % 9 == 0: break
plt.show()

In [9]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Define MobileNetV2
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.5), # Added to fix the 12% overfitting gap
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.0001, momentum=0.9),
              loss='categorical_crossentropy', metrics=['accuracy'])

# Data Generators
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255, validation_split=0.2, rotation_range=20,
    horizontal_flip=True, zoom_range=0.2, brightness_range=[0.8, 1.2]
)

train_gen = datagen.flow_from_directory(processed_base, target_size=(224, 224), batch_size=8, subset='training')
val_gen = datagen.flow_from_directory(processed_base, target_size=(224, 224), batch_size=8, subset='validation')

# Execute Training
history = model.fit(train_gen, validation_data=val_gen, epochs=100, callbacks=[checkpoint])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Found 933 images belonging to 3 classes.
Found 232 images belonging to 3 classes.
Epoch 1/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step - accuracy: 0.4859 - loss: 1.1398
Epoch 1: val_accuracy improved from None to 0.43103, saving model to /content/drive/MyDrive/Dermascan_Project/acne_model_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Dermascan_Project/acne_model_best.keras
117/117 ━━━━━━━━━━━━━━━━━━━━ 45s 354ms/step - accuracy: 0.5080 - loss: 1.1049 - val_accuracy: 0.4310 - val_loss: 0.9585
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step - accuracy: 0.4881 - loss: 1.0775
Epoch 2: val_accuracy improved from 0.43103 to 0.53879, saving model to /content/drive/MyDrive/Dermascan_Project/acne_model_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Dermascan_Project/acne_model_best.keras
117/117 ━━━━━━━━━━━━━━━━━━━━ 37s 314ms/step - accuracy: 0.5080 - loss: 1.0381 - val_accuracy: 0.5388 - val_l

In [10]:
# Matching the research paper's exact settings [cite: 223]
#optimizer = tf.keras.optimizers.SGD(learning_rate=0.0001, momentum=0.9)

#model.compile(
 #   optimizer=optimizer,
  #  loss='categorical_crossentropy',
   # metrics=['accuracy']
#)

#print("Hyperparameters set: SGD Optimizer | LR=0.0001 | Momentum=0.9")

Hyperparameters set: SGD Optimizer | LR=0.0001 | Momentum=0.9


In [11]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# This setup follows the paper's augmentation strategy [cite: 167, 172]
#datagen = ImageDataGenerator(
 #   rescale=1./255,          # Normalizes pixels
  #  validation_split=0.2,    # 20% validation split as per the paper [cite: 90]
   # rotation_range=20,       # Random rotation [cite: 168]
   # horizontal_flip=True,    # Mirroring images [cite: 168]
   # zoom_range=0.2,          # Zooming for scale variation [cite: 169]
    #brightness_range=[0.8, 1.2] # Adjusting for lighting [cite: 169]
#)

# Creating the Training Generator
#train_gen = datagen.flow_from_directory(
 #   'Organized_Acne_Data',
  #  target_size=(224, 224),  # Standard input for MobileNetV2 [cite: 211, 436]
   # batch_size=8,            # Specified in Table II [cite: 223]
    #class_mode='categorical',
    #subset='training'
#)

# Creating the Validation Generator
#val_gen = datagen.flow_from_directory(
 #   'Organized_Acne_Data',
  #  target_size=(224, 224),
   # batch_size=8,
    #class_mode='categorical',
    #subset='validation'
#)

Found 933 images belonging to 3 classes.
Found 232 images belonging to 3 classes.


In [12]:
# This will now recognize the generators
#history = model.fit(
 #   train_gen,
  #  epochs=100,              # Based on Table II [cite: 223, 344]
   # validation_data=val_gen, # Validating against the 20% split [cite: 90]
    #callbacks=[checkpoint],  # Saves progress to your Google Drive
    #verbose=1
#)

Epoch 1/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step - accuracy: 0.7190 - loss: 0.5632
Epoch 1: val_accuracy did not improve from 0.71552
117/117 ━━━━━━━━━━━━━━━━━━━━ 42s 323ms/step - accuracy: 0.7085 - loss: 0.5613 - val_accuracy: 0.6552 - val_loss: 0.7117
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step - accuracy: 0.7319 - loss: 0.5533
Epoch 2: val_accuracy did not improve from 0.71552
117/117 ━━━━━━━━━━━━━━━━━━━━ 37s 313ms/step - accuracy: 0.7449 - loss: 0.5450 - val_accuracy: 0.6897 - val_loss: 0.6605
Epoch 3/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step - accuracy: 0.7092 - loss: 0.5986
Epoch 3: val_accuracy did not improve from 0.71552
117/117 ━━━━━━━━━━━━━━━━━━━━ 37s 312ms/step - accuracy: 0.7213 - loss: 0.5783 - val_accuracy: 0.6552 - val_loss: 0.6818
Epoch 4/100
 92/117 ━━━━━━━━━━━━━━━━━━━━ 6s 265ms/step - accuracy: 0.7011 - loss: 0.5936

KeyboardInterrupt: 

In [ ]:
#import tensorflow as tf
#import os

# 1. Load your existing .keras model from Google Drive
# Ensure the filename matches what you have in your 'Dermascan_Project' folder
#model_path = os.path.join(BASE_PATH, 'acne_model_best.keras')

#if os.path.exists(model_path):
 #   print("📂 Loading existing .keras model...")
  #  model = tf.keras.models.load_model(model_path)

    # 2. Initialize the TFLite Converter
  #  converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # 3. Apply Optimization (Reduces file size for Android)
  #  converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # 4. Convert and Save
   # tflite_model = converter.convert()
   # tflite_path = os.path.join(BASE_PATH, 'acne_model_final.tflite')

   # with open(tflite_path, "wb") as f:
    #    f.write(tflite_model)

  # print(f"✅ Success! Your Android-ready model is at: {tflite_path}")
#else:
 #   print(f"❌ Error: Could not find {model_path}. Please check your BASE_PATH.")